In [ ]:
# Cell 1: Environment Setup, Dependencies & Repository Cloning
import os
import sys
import subprocess

print("[SETUP] Setting up Kaggle CPU Data Generator Environment...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "sympy"], check=True)

repo_url = "https://github.com/dsainvg001/transformer-math.git"
repo_dir = "transformer-math"

if not os.path.exists(repo_dir):
    print(f"[GIT] Cloning repository from {repo_url}...")
    subprocess.run(["git", "clone", repo_url], check=True)
else:
    print(f"[GIT] Repository {repo_dir} already present.")

if os.path.exists(repo_dir):
    os.chdir(repo_dir)
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

print(f"[PATH] Current Directory: {os.getcwd()}")

In [ ]:
# Cell 2: Hugging Face Authentication Token Configuration
from getpass import getpass

hf_token = os.environ.get("HFTOKEN") or os.environ.get("HF_TOKEN")

# Check for Kaggle Secrets if running inside Kaggle Notebook
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret("HFTOKEN") or user_secrets.get_secret("HF_TOKEN")
    except Exception:
        pass

# Fallback to interactive input if token is not set in environment or secrets
if not hf_token:
    print("[AUTH] HFTOKEN not detected in environment or Kaggle Secrets.")
    hf_token = getpass("Enter your Hugging Face Write Token: ")

os.environ["HFTOKEN"] = hf_token
print(f"[AUTH] Hugging Face Token Configured (Length: {len(hf_token)} chars).")

In [ ]:
# Cell 3: Dataset Generation Settings (Kaggle CPU Session)
DEBUG_MODE = False  # Set to True for a 1-minute 10k test run

REPO_ID = "durgasai299792458/mathmetics-dataset"
SHARD_SIZE = 100000  # 100,000 data points per shard
NUM_SAMPLES = 250000000 if not DEBUG_MODE else 10000
OUTPUT_DIR = "/kaggle/working/hf_shards"

import multiprocessing as mp
num_workers = mp.cpu_count()

print("=========================================================================")
print(f"[CONFIG] Kaggle CPU Session Dataset Generator")
print("=========================================================================")
print(f"- Target Dataset Repo: {REPO_ID}")
print(f"- Total Target Samples: {NUM_SAMPLES}")
print(f"- Shard Size: {SHARD_SIZE} samples/shard")
print(f"- CPU Cores Available: {num_workers}")
print(f"- Output Directory: {OUTPUT_DIR}")
print(f"- Debug Mode: {DEBUG_MODE}")
print("=========================================================================")

In [ ]:
# Cell 4: Launch High-Throughput 250M Dataset Generation & Sharded Upload
cmd = [
    sys.executable, "generate_dataset.py",
    "--num-samples", str(NUM_SAMPLES),
    "--shard-size", str(SHARD_SIZE),
    "--repo-id", REPO_ID,
    "--output-dir", OUTPUT_DIR,
    "--num-workers", str(num_workers)
]

if DEBUG_MODE:
    cmd.append("--debug")

print(f"[EXECUTE] Running: {' '.join(cmd)}", flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

# Stream output logs in real-time
for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()
if proc.returncode == 0:
    print(f"\n[SUCCESS] Dataset successfully generated and uploaded to https://huggingface.co/datasets/{REPO_ID}")
else:
    print(f"\n[ERROR] Generation script exited with return code {proc.returncode}")